In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import itertools

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [71]:
file_path ='/content/drive/MyDrive/MASTER/TESI/dati/definitivi/final_graph_8maggio26_1.tsv'
try:
    grafo = pd.read_csv(file_path, sep = "\t")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

In [72]:
del grafo['Unnamed: 0']

In [73]:
G = nx.from_pandas_edgelist(grafo, 'ID1', 'ID2', edge_attr=['Edge_PDB', 'Edge_UP', 'TM-score'])
rows = []
for i, cc in enumerate(nx.connected_components(G)):
  for node in cc:
    rows.append({"node": node, "CC_ID": i})

cc_df = pd.DataFrame(rows)
#cc_df.to_csv("connected_components.tsv", sep="\t", index=False)

In [75]:
cc_tot_ids = {}
cc_ids = {}
ccs = list(set(cc_df['CC_ID']))
for cc in ccs:
  nodes = list(cc_df[cc_df['CC_ID']==cc]['node'])
  cc_tot_ids[cc]=len(nodes)
  cc_ids[cc]=','.join(nodes)

In [76]:
cc_tot_pdb = {}
cc_tot_up = {}
for cc in ccs:
  cc_tot_pdb[cc]=0
  cc_tot_up[cc]=0
for cc in ccs:
  nodes = list(cc_df[cc_df['CC_ID']==cc]['node'])
  for id in nodes:
    id1_df = grafo[grafo['ID1']==id].reset_index(drop=True)
    for i in range(id1_df.shape[0]):
      if id1_df['Edge_PDB'][i]=='p':
        cc_tot_pdb[cc]+=1
      elif id1_df['Edge_UP'][i]=='u':
        cc_tot_up[cc]+=1
    id2_df = grafo[grafo['ID1']==id].reset_index(drop=True)
    for i in range(id2_df.shape[0]):
      if id2_df['Edge_PDB'][i]=='p':
        cc_tot_pdb[cc]+=1
      elif id2_df['Edge_UP'][i]=='u':
        cc_tot_up[cc]+=1

In [77]:
nodes = []
tot_nodes = []
tot_pdb = []
tot_up = []
for cc in ccs:
  nodes.append(cc_ids[cc])
  tot_nodes.append(cc_tot_ids[cc])
  tot_pdb.append(cc_tot_pdb[cc])
  tot_up.append(cc_tot_up[cc])

In [78]:
file_path ='/content/drive/MyDrive/MASTER/TESI/dati/mCSM-PPI2.csv'
try:
    s4169 = pd.read_csv(file_path, sep = "\t")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

In [79]:
pdb = s4169['pdb'].to_numpy()
chain = pdb + '_' + s4169['mutation'].str[1]
s4169['chain']=chain

In [80]:
srv_chain = s4169['chain'].value_counts().to_dict()

In [81]:
srvs = []
i = 0
for cc in ccs:
  chains = cc_ids[cc].split(',')
  tot = 0
  for c in chains:
    i += 1
    if c in srv_chain:
      tot += srv_chain[c]
  srvs.append(tot)

In [83]:
cc_analysis_df = pd.DataFrame({'CC_ID': ccs, 'SRVs': srvs, 'PDB_arches': tot_pdb, 'Uniprot_arches': tot_up, 'Tot_nodes': tot_nodes, 'Nodes': nodes})

In [70]:
cc_analysis_df.to_csv('CC_analysis.tsv', sep = '\t')